# Therapy-Agent Walkthrough: Ekterly/HAE Case

This notebook steps through the therapy-agent pipeline for the Hereditary Angioedema (HAE) case,
showing how each node reasons about SERPING1 loss-of-function and arrives at KLKB1 inhibition
as the therapeutic strategy — citing sebetralstat (Ekterly), FDA approved July 2025.

## Background

**Gene:** SERPING1 (C1-esterase inhibitor, C1-INH)

**Disease:** Hereditary angioedema (HAE) — recurrent attacks of subcutaneous and mucosal edema

**Mechanism:** SERPING1 haploinsufficiency → uncontrolled plasma kallikrein (KLKB1) → excess bradykinin → vascular permeability

**Drug:** sebetralstat (Ekterly, KalVista) — oral KLKB1 inhibitor — FDA approved July 2025 (KONFIDENT Phase 3 trial)

In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import asyncio
from therapy_agent.state import make_initial_state

assert os.environ.get('ANTHROPIC_API_KEY'), 'Set ANTHROPIC_API_KEY first'
print('Setup complete')

## Step 1: parse_input

Extract structured information from free-text input.

In [ ]:
from therapy_agent.nodes.parse_input import parse_input_node

state = make_initial_state(
    gene='SERPING1',
    mutation='frameshift or large deletion causing haploinsufficiency of C1 esterase inhibitor',
    disease_phenotype='hereditary angioedema (HAE) with recurrent subcutaneous and mucosal edema attacks',
)

result = asyncio.run(parse_input_node(state))
print('gene_symbol:', result.get('gene_symbol'))
print('mutation_type:', result.get('mutation_type'))
print('phenotype_terms:', result.get('phenotype_terms'))
print('trace:', result.get('reasoning_trace', []))

## Step 2: mechanism_classifier

Classify the molecular mechanism using Claude with few-shot prompting.

In [ ]:
from therapy_agent.nodes.mechanism_classifier import mechanism_classifier_node

state2 = dict(state)
state2.update(result)

mech_result = asyncio.run(mechanism_classifier_node(state2))
print('Mechanism:', mech_result.get('molecular_mechanism'))
print('Confidence:', mech_result.get('mechanism_confidence'))
print('Reasoning:', mech_result.get('mechanism_reasoning'))

## Step 3: pathway_expansion

Expand the pathway using Reactome (curated fallback for SERPING1).

In [ ]:
from therapy_agent.nodes.pathway_expansion import pathway_expansion_node

state3 = dict(state2)
state3.update(mech_result)

pathway_result = asyncio.run(pathway_expansion_node(state3))
print('Pathway genes:', pathway_result.get('pathway_genes'))
print('Context:', pathway_result.get('pathway_context'))

## Step 4: druggable_target_search

Search ChEMBL and DrugBank for druggable targets in the pathway.

In [ ]:
from therapy_agent.nodes.druggable_target_search import druggable_target_search_node
import json

state4 = dict(state3)
state4.update(pathway_result)

drug_result = asyncio.run(druggable_target_search_node(state4))
print('Candidate targets:')
for t in drug_result.get('candidate_targets', []):
    print(f"  {t['gene_name']}: {[d['name'] for d in t.get('drugbank_drugs', [])]}")

## Step 5: strategy_synthesis

Claude synthesizes a therapeutic strategy given the mechanism, pathway, and druggable targets.

In [ ]:
from therapy_agent.nodes.strategy_synthesis import strategy_synthesis_node

state5 = dict(state4)
state5.update(drug_result)

strat_result = asyncio.run(strategy_synthesis_node(state5))
strat = strat_result.get('strategy')
if strat:
    print('Target:', strat.get('target_protein'))
    print('Pathway:', strat.get('target_pathway'))
    print('Modulation:', strat.get('modulation_type'))
    print('Confidence:', strat.get('confidence_score'))
    print('Drugs:', strat.get('precedent_drugs'))
    print('Citations:', strat.get('citations'))
    print('Rationale:', strat.get('rationale', '')[:200])

## Step 6: self_critique

Claude reviews its own output for unsupported claims and adjusts confidence.

In [ ]:
from therapy_agent.nodes.self_critique import self_critique_node

state6 = dict(state5)
state6.update(strat_result)

critique_result = asyncio.run(self_critique_node(state6))
print('Critique notes:', critique_result.get('critique_notes'))
final = critique_result.get('final_strategy')
if final:
    print('Final confidence:', final.get('confidence_score'))

## Full Pipeline

Run the complete LangGraph pipeline end-to-end.

In [ ]:
from therapy_agent.graph import run_agent

final_state = asyncio.run(run_agent(
    gene='SERPING1',
    mutation='frameshift causing haploinsufficiency of C1 esterase inhibitor',
    disease_phenotype='hereditary angioedema',
))

strategy = final_state.get('final_strategy') or final_state.get('strategy')
print('=== FINAL STRATEGY ===')
if strategy:
    for k, v in strategy.items():
        print(f'{k}: {v}')

print('\n=== CITATIONS ===')
for c in final_state.get('citations', []):
    print(' -', c)